# Telecom X — EDA (Parte 1)

Este notebook faz a **análise exploratória** usando o dataset processado pelo ETL (`data/processed/telecom_transformed.parquet`).

**Data:** 05/03/2026

> Se o arquivo processado não existir, rode antes:

```bash
python scripts/run_etl.py
```


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1) Carregar dataset processado

In [ ]:
processed_parquet = "data/processed/telecom_transformed.parquet"
processed_csv = "data/processed/telecom_transformed.csv"

if os.path.exists(processed_parquet):
    df = pd.read_parquet(processed_parquet)
    origem = processed_parquet
elif os.path.exists(processed_csv):
    df = pd.read_csv(processed_csv)
    origem = processed_csv
else:
    raise FileNotFoundError(
        "Não encontrei o dataset processado. Rode antes: python scripts/run_etl.py"
    )

print("Carregado:", origem)
print("Shape:", df.shape)
display(df.head(5))

## 2) Visão geral

Tipos, faltantes e valores únicos (amostra).

In [ ]:
display(df.info())

faltantes = df.isna().mean().sort_values(ascending=False)
display((faltantes*100).round(2).to_frame("% faltantes").head(25))

# colunas com muitos valores únicos (candidatas a ID)
nunique = df.nunique(dropna=True).sort_values(ascending=False)
display(nunique.head(15).to_frame("n_unique"))

## 3) Distribuição do churn

O dataset da Telecom X costuma ter uma coluna `churn` (geralmente 'Yes/No').

In [ ]:
if "churn" not in df.columns:
    raise KeyError("Coluna 'churn' não encontrada. Verifique o ETL/arquivo de origem.")

churn_counts = df["churn"].value_counts(dropna=False)
churn_prop = df["churn"].value_counts(normalize=True, dropna=False) * 100

display(pd.DataFrame({"contagem": churn_counts, "proporção_%": churn_prop.round(2)}))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="churn")
plt.title("Distribuição do Churn")
plt.tight_layout()
plt.show()

## 4) Churn por variáveis categóricas

Aqui é onde geralmente aparecem padrões fortes (ex.: contrato, internet, pagamento).

In [ ]:
def plot_churn_por_categoria(col):
    if col not in df.columns:
        print(f"Coluna '{col}' não existe no dataset.")
        return
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, x=col, hue="churn")
    plt.title(f"Churn por {col}")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    tab = pd.crosstab(df[col], df["churn"], normalize="index") * 100
    display(tab.round(2))

# colunas mais comuns no dataset do desafio (podem variar)
candidatas = ["contract_type", "internetservice", "paymentmethod", "dependents", "partner"]
for c in candidatas:
    if c in df.columns:
        plot_churn_por_categoria(c)

## 5) Variáveis numéricas × churn

Boxplots ajudam a ver diferenças de distribuição entre quem evadiu e quem não evadiu.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
print("Numéricas:", num_cols)

# Algumas colunas comuns no dataset Telco
candidatas_num = [c for c in ["monthlycharges", "totalcharges", "customer_tenure"] if c in df.columns]
if not candidatas_num and num_cols:
    candidatas_num = num_cols[:3]  # fallback

for col in candidatas_num:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x="churn", y=col)
    plt.title(f"{col} por churn")
    plt.tight_layout()
    plt.show()

    display(df.groupby("churn")[col].describe().round(2))

## 6) Correlação (numéricas)

Matriz de correlação para variáveis numéricas e tabela com as maiores correlações absolutas.

In [ ]:
num_df = df.select_dtypes(include="number")
if num_df.shape[1] >= 2:
    corr = num_df.corr(numeric_only=True)

    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=False, cmap="coolwarm")
    plt.title("Matriz de correlação (numéricas)")
    plt.tight_layout()
    plt.show()

    # top correlações absolutas (pares)
    corr_abs = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    top_pairs = corr_abs.stack().sort_values(ascending=False).head(15)
    display(top_pairs.to_frame("correlação_abs"))
else:
    print("Poucas colunas numéricas para correlação.")

## 7) Gerar figuras e relatório automaticamente

Este bloco cria a pasta `figs/` e salva algumas imagens + escreve um relatório em `relatorio_telecomx.md` com métricas básicas.

> Isso resolve a cobrança de **EDA + evidências + insights** em um arquivo separado.

In [ ]:
import os
os.makedirs("figs", exist_ok=True)

# Fig 1: churn por contract_type (se existir)
if "contract_type" in df.columns:
    plt.figure(figsize=(8,5))
    sns.countplot(data=df, x="contract_type", hue="churn")
    plt.title("Churn por tipo de contrato")
    plt.tight_layout()
    plt.savefig("figs/churn_por_contrato.png", dpi=150)
    plt.close()

# Fig 2: boxplot monthlycharges
if "monthlycharges" in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x="churn", y="monthlycharges")
    plt.title("Monthly Charges por churn")
    plt.tight_layout()
    plt.savefig("figs/monthlycharges_boxplot.png", dpi=150)
    plt.close()

# Fig 3: correlação
if num_df.shape[1] >= 2:
    plt.figure(figsize=(10,8))
    sns.heatmap(num_df.corr(numeric_only=True), annot=False, cmap="coolwarm")
    plt.title("Matriz de correlação (numéricas)")
    plt.tight_layout()
    plt.savefig("figs/correlation_heatmap.png", dpi=150)
    plt.close()

# Métricas para o relatório
churn_rate = (df["churn"].value_counts(normalize=True) * 100).to_dict()

insights = []
insights.append(f"- Proporção de churn: { {k: round(v,2) for k,v in churn_rate.items()} }")

if "contract_type" in df.columns:
    tab = pd.crosstab(df["contract_type"], df["churn"], normalize="index")*100
    # tenta pegar a taxa de 'Yes' se existir
    yes_col = next((c for c in tab.columns if str(c).lower() in ["yes","sim","1","true"]), None)
    if yes_col is not None:
        ranking = tab[yes_col].sort_values(ascending=False).head(3)
        insights.append("- Top 3 contratos com maior churn (%): " + ", ".join([f"{idx}: {val:.2f}%" for idx,val in ranking.items()]))
    else:
        insights.append("- Churn por contrato: veja a tabela/figura gerada.")

if "monthlycharges" in df.columns:
    med = df.groupby("churn")["monthlycharges"].median().to_dict()
    insights.append(f"- Mediana de monthlycharges por churn: { {k: round(v,2) for k,v in med.items()} }")

# escreve relatório
rel = []
rel.append("# Relatório — Telecom X (Parte 1: ETL + EDA)")
rel.append("")
rel.append(f"Data: {pd.Timestamp.today().date()}")
rel.append("")
rel.append("## Principais métricas")
rel.append("")
rel.extend(insights)
rel.append("")
rel.append("## Figuras")
rel.append("")
rel.append("- Churn por contrato: `figs/churn_por_contrato.png`")
rel.append("- MonthlyCharges por churn: `figs/monthlycharges_boxplot.png`")
rel.append("- Correlação (numéricas): `figs/correlation_heatmap.png`")
rel.append("")
rel.append("## Observações finais (para você completar)")
rel.append("")
rel.append("- Quais grupos de clientes parecem mais propensos a cancelar?")
rel.append("- Que ações de retenção você sugere com base nos padrões observados?")
rel.append("")

with open("relatorio_telecomx.md", "w", encoding="utf-8") as f:
    f.write("\n".join(rel))

print("✅ Figuras geradas em: figs/")
print("✅ Relatório gerado em: relatorio_telecomx.md")